# vLLM vs. SGLang vs. TensorRT-LLM: Comparing LLM Serving Architectures

The full code of [vLLM vs. SGLang vs. TensorRT-LLM: Comparing LLM Serving Architectures](https://residual-stream.lvtan71.workers.dev/posts/vllm-vs-sglang-vs-tensorrt-llm/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/vllm-vs-sglang-vs-tensorrt-llm) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vllm-vs-sglang-vs-tensorrt-llm.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 2 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "transformers==5.19.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "vllm-vs-sglang-vs-tensorrt-llm"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/vllm-vs-sglang-vs-tensorrt-llm


## `reuse.py`

Replays multi-turn ShareGPT chat through each engine's default cache at three cache sizes, then through whole blocks of four sizes.

In [2]:
!{sys.executable} reuse.py

2161 requests from 429 multi-turn conversations; 1,583,998 prompt tokens, 733 per request
prompt tokens reused, each engine's default match
cache size     SGLang (any prefix)   vLLM (16-token blocks)   TensorRT-LLM (32, partial reuse)


    65,536             80.5%                 79.4%                    77.5%


   262,144             92.0%                 90.9%                    92.0%


 unlimited             92.0%                 90.9%                    92.0%
whole blocks only, by block size (tokens)
cache size           16      32      64     128


    65,536        79.4%   77.7%   76.6%   74.2%


   262,144        90.9%   89.6%   87.8%   84.2%


 unlimited        90.9%   89.6%   87.8%   84.2%


The post printed:

```text
2161 requests from 429 multi-turn conversations; 1,583,998 prompt tokens, 733 per request
prompt tokens reused, each engine's default match
cache size     SGLang (any prefix)   vLLM (16-token blocks)   TensorRT-LLM (32, partial reuse)
    65,536             80.5%                 79.4%                    77.5%
   262,144             92.0%                 90.9%                    92.0%
 unlimited             92.0%                 90.9%                    92.0%
whole blocks only, by block size (tokens)
cache size           16      32      64     128
    65,536        79.4%   77.7%   76.6%   74.2%
   262,144        90.9%   89.6%   87.8%   84.2%
 unlimited        90.9%   89.6%   87.8%   84.2%
```